# Comparativa de Modelos Base (Pruebas y Validación — Capítulo 7)

Este notebook consolida tres experimentos comparativos independientes de las Fases 1–4: (1) visión artificial, OpenAI CLIP vs. StreetCLIP; (2) tolerancia lingüística del motor de texto ante distintos estilos de redacción y niveles de ruido; y (3) búsqueda espacial, BallTree vs. KDTree vs. fuerza bruta, para el componente geográfico del sistema.

El experimento de visión y el de búsqueda espacial usan métricas independientes del umbral de decisión, de forma de poder comparar modelos con distinta calibración sin favorecer a ninguno. El experimento de NLP evalúa el pipeline real de clasificación de texto (mapa semántico de anclas + normalizador de producción) sobre el conjunto de 588 reportes válidos del dataset descripto en el Anexo A, agrupados según su estilo de redacción y su nivel de ruido tipográfico.

In [1]:
!pip install -q transformers torch scikit-learn pandas matplotlib seaborn pillow tqdm sentence-transformers

In [2]:
import os
import time
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch
from transformers import CLIPProcessor, CLIPModel
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import BallTree, KDTree
from tqdm import tqdm

from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/validation_dataset"
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Dispositivo: {device}")
from sklearn.metrics import roc_auc_score, roc_curve
import math

def wilson(k, n, z=1.96):
    if n == 0:
        return (0.0, 0.0)
    p = k / n; den = 1 + z*z/n
    centro = (p + z*z/(2*n)) / den
    marg = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / den
    return (centro - marg, centro + marg)

def fmt_ci(k, n):
    if n == 0:
        return 'n/d'
    lo, hi = wilson(k, n)
    return f'{k}/{n} = {k/n*100:.1f}% (IC95% {lo*100:.1f}-{hi*100:.1f}%)'


Mounted at /content/drive
Dispositivo: cuda


## 1. Visión Artificial: OpenAI CLIP vs. StreetCLIP
Evaluación del filtro visual sobre las 714 imágenes del dataset. Se reportan (a) métricas con los umbrales fijos de producción (calibrados originalmente para OpenAI CLIP) y (b) métricas independientes del umbral (AUC, TPR a TNR ≥ 95 %, accuracy de categoría sin ningún filtro), que permiten una comparación equitativa entre modelos con distinta calibración.

In [ ]:
print("Cargando dataset canónico...")
with open(os.path.join(BASE_DIR, "real_dataset.json"), "r", encoding="utf-8") as f:
    dataset = json.load(f)
n_desc_total = sum(1 for d in dataset if d["expected_category"] == "Descarte")
print(f"{len(dataset)} muestras: {len(dataset) - n_desc_total} válidos + {n_desc_total} descartes")

print("Cargando modelos (tardará unos minutos)...")
clip_name = "openai/clip-vit-base-patch32"
proc_openai = CLIPProcessor.from_pretrained(clip_name)
clip_openai = CLIPModel.from_pretrained(clip_name).to(device).eval()
street_name = "geolocal/StreetCLIP"
proc_street = CLIPProcessor.from_pretrained(street_name)
clip_street = CLIPModel.from_pretrained(street_name).to(device).eval()

real_photo_labels = ["a real photograph of an outdoor urban environment", "a digital illustration, drawing, or meme"]
outdoor_labels = ["an outdoor urban street scene in a city", "an indoor scene, a person, a pet, food or a natural landscape"]
problem_labels = [
    "a blocked or flooded drainage ditch or canal on the street",
    "a broken or unlit streetlight or fallen electric pole",
    "a fallen tree, dangerous branches or roots lifting the sidewalk",
    "a pothole, damaged pavement or broken road surface",
    "garbage, waste, rubble or trash accumulated on the street",
    "a damaged bench, broken playground or neglected public park",
    "a broken traffic light, fallen road sign or faded road markings",
    "a broken sidewalk, missing tiles or blocked pedestrian access",
    "a water leak, broken pipe or overflowing sewer on the street",
    "a normal street or public space in good condition with no issues",
    "an unrelated scene with no urban infrastructure problems visible",
    "explicitly inappropriate, offensive, or unsafe content",   # idéntico a la Fase 4 del sistema
]
label_to_category = {
    problem_labels[0]: "Acequias y Drenajes", problem_labels[1]: "Alumbrado Público",
    problem_labels[2]: "Arbolado Público", problem_labels[3]: "Baches y Pavimentación",
    problem_labels[4]: "Limpieza y Residuos", problem_labels[5]: "Plazas y Parques",
    problem_labels[6]: "Semáforos y Señalización", problem_labels[7]: "Veredas y Accesibilidad",
    problem_labels[8]: "Agua y Cloacas",
}
NO_PROBLEM_IDX = [9, 10, 11]

def _sync():
    if device == "cuda":
        torch.cuda.synchronize()

def extraer_scores(model, processor, model_name):
    """Una fila por imagen con todas las probabilidades (los umbrales se aplican después)."""
    tx = {k: {kk: vv.to(device) for kk, vv in processor(text=lbl, return_tensors="pt", padding=True).items()}
          for k, lbl in [("real", real_photo_labels), ("out", outdoor_labels), ("prob", problem_labels)]}

    def tres_pasadas(img_inputs):
        with torch.no_grad():
            r = model(**{**img_inputs, **tx["real"]}).logits_per_image.softmax(dim=1)[0]
            o = model(**{**img_inputs, **tx["out"]}).logits_per_image.softmax(dim=1)[0]
            p = model(**{**img_inputs, **tx["prob"]}).logits_per_image.softmax(dim=1)[0]
        return r, o, p

    # Calentamiento (evita contaminar la latencia con la inicialización de CUDA)
    dummy = {k: v.to(device) for k, v in processor(images=Image.new("RGB", (224, 224)), return_tensors="pt").items()}
    for _ in range(3):
        tres_pasadas(dummy)
    _sync()

    filas, omitidas = [], 0
    for item in tqdm(dataset, desc=f"Evaluando {model_name}"):
        img_path = os.path.join(BASE_DIR, item["image_path"])
        if not os.path.exists(img_path):
            omitidas += 1; continue
        try:
            img = Image.open(img_path).convert("RGB")
        except Exception:
            omitidas += 1; continue
        img_inputs = {k: v.to(device) for k, v in processor(images=img, return_tensors="pt").items()}
        _sync(); t0 = time.perf_counter()
        r, o, p = tres_pasadas(img_inputs)
        pr = p.cpu().numpy(); _sync()
        lat = (time.perf_counter() - t0) * 1000
        scores = {problem_labels[i]: float(pr[i]) for i in range(len(problem_labels))}
        validas = {k: v for k, v in scores.items() if k not in [problem_labels[i] for i in NO_PROBLEM_IDX]}
        best_lbl = max(validas, key=validas.get)
        filas.append({
            "esperado": item["expected_category"],
            "prob_real": float(r[0]), "prob_outdoor": float(o[0]),
            "p_no_problem": max(scores[problem_labels[9]], scores[problem_labels[10]]),
            "p_inapp": scores[problem_labels[11]],
            "best_prob": validas[best_lbl], "sug_cat": label_to_category[best_lbl], "lat_ms": lat,
        })
    df = pd.DataFrame(filas)
    print(f"{model_name}: {len(df)} evaluadas, {omitidas} omitidas (imagen ausente o ilegible)")
    return df

def resumir(df, model_name):
    es_desc = df["esperado"] == "Descarte"
    rechazo = ((df.prob_real < 0.55) | (df.prob_outdoor < 0.50) | (df.p_inapp > 0.40)
               | (df.p_no_problem > 0.40) | (df.best_prob < 0.32))          # umbrales de producción (Fase 4)
    nd, nv = int(es_desc.sum()), int((~es_desc).sum())
    tn, tp = int((rechazo & es_desc).sum()), int((~rechazo & ~es_desc).sum())
    aceptados = ~rechazo & ~es_desc
    acc_cond = (df.sug_cat[aceptados] == df.esperado[aceptados]).mean() * 100 if aceptados.any() else 0
    acc_top1 = (df.sug_cat[~es_desc] == df.esperado[~es_desc]).mean() * 100          # sin ningún filtro
    y = (~es_desc).astype(int)
    comb = df.prob_real * df.prob_outdoor * df.best_prob * (1 - np.maximum(df.p_no_problem, df.p_inapp))
    fpr, tpr, _ = roc_curve(y, comb)
    return {
        "Modelo": model_name,
        "TNR descartes (umbrales fijos)": fmt_ci(tn, nd),
        "TPR válidos (umbrales fijos)": fmt_ci(tp, nv),
        "Acc. categoría | aceptados %": round(acc_cond, 1),
        "Acc. top-1 sin filtro %": round(acc_top1, 1),
        "AUC (score combinado)": round(roc_auc_score(y, comb), 3),
        "AUC (best_prob)": round(roc_auc_score(y, df.best_prob), 3),
        "TPR @ TNR>=95%": round(float(tpr[fpr <= 0.05].max()) * 100, 1),
        "Latencia media (ms)": round(df.lat_ms.mean(), 1),
        "Latencia p95 (ms)": round(df.lat_ms.quantile(0.95), 1),
    }

df_openai = extraer_scores(clip_openai, proc_openai, "OpenAI CLIP (ViT-B/32)")
df_street = extraer_scores(clip_street, proc_street, "StreetCLIP (ViT-L/14)")
df_comp_vision = pd.DataFrame([resumir(df_openai, "OpenAI CLIP (ViT-B/32)"), resumir(df_street, "StreetCLIP (ViT-L/14)")])
display(df_comp_vision.T)
df_comp_vision.to_csv(os.path.join(BASE_DIR, "comparativa_vision.csv"), index=False)
df_openai.to_csv(os.path.join(BASE_DIR, "scores_vision_openai.csv"), index=False)
df_street.to_csv(os.path.join(BASE_DIR, "scores_vision_streetclip.csv"), index=False)
print("[OK] Guardado en", BASE_DIR)
print("Lectura: si StreetCLIP tiene AUC/TPR@TNR95 similares a OpenAI CLIP pero peor TPR con umbrales fijos, "
      "la 'caída de sensibilidad' es un problema de calibración de umbrales, no de capacidad del modelo.")


Cargando dataset canónico...
714 muestras: 588 válidos + 126 descartes
Cargando modelos (tardará unos minutos)...


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

...

OpenAI CLIP (ViT-B/32): 714 evaluadas, 0 omitidas (imagen ausente o ilegible)


Evaluando StreetCLIP (ViT-L/14): 100%|██████████| 714/714 [05:09<00:00,  2.31it/s]

StreetCLIP (ViT-L/14): 714 evaluadas, 0 omitidas (imagen ausente o ilegible)


,0,1
Modelo,OpenAI CLIP (ViT-B/32),StreetCLIP (ViT-L/14)
TNR descartes (umbrales fijos),123/126 = 97.6% (IC95% 93.2-99.2%),122/126 = 96.8% (IC95% 92.1-98.8%)
TPR válidos (umbrales fijos),526/588 = 89.5% (IC95% 86.7-91.7%),532/588 = 90.5% (IC95% 87.8-92.6%)
Acc. categoría | aceptados %,80.4,84.6
Acc. top-1 sin filtro %,80.8,85.2
AUC (score combinado),0.994,0.994
AUC (best_prob),0.99,0.987
TPR @ TNR>=95%,97.1,95.9
Latencia media (ms),80.6,404.4
Latencia p95 (ms),210.5,422.2


[OK] Guardado en /content/drive/MyDrive/validation_dataset
Lectura: si StreetCLIP tiene AUC/TPR@TNR95 similares a OpenAI CLIP pero peor TPR con umbrales fijos, la 'caída de sensibilidad' es un problema de calibración de umbrales, no de capacidad del modelo.


## 2. Tolerancia Lingüística (NLP)
Se evalúa el pipeline real de clasificación de texto (similitud contra el mapa semántico de anclas, categoría de mayor similitud, sin aplicar umbral de aceptación) con el normalizador de producción, sobre los 588 textos válidos del dataset, agrupados según su estilo de redacción y su nivel de ruido tipográfico (cada texto pertenece a un único grupo: Claro, Indirecto, Formal, Ruido medio/fuerte o Vago). Se comparan dos modelos de embeddings semánticos (sentence-transformer en español derivado de BETO, y MiniLM multilingüe, cada uno con y sin normalizador) contra dos baselines léxicos de TF-IDF (por palabra y por n-gramas de caracteres), ambos ajustados únicamente con las anclas del mapa semántico, sin ver los textos de evaluación.

In [4]:
import os
import time
import json
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, util
from IPython.display import display

# Expresa cada resultado como conteo, porcentaje e intervalo de confianza de Wilson al 95%.
def fmt_ci(k, n):
    from statsmodels.stats.proportion import proportion_confint
    if n == 0: return "0/0"
    low, high = proportion_confint(k, n, alpha=0.05, method='wilson')
    return f"{k}/{n} = {k/n*100:.1f}% (IC95% {low*100:.1f}-{high*100:.1f}%)"

print("Cargando modelos NLP...")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
beto_model = SentenceTransformer('hiiamsid/sentence_similarity_spanish_es').to(device)
minilm_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2').to(device)

semantic_map = {
    "Acequias y Drenajes": ["acequia", "drenaje", "canal", "cuneta", "agua estancada", "puente roto", "zanja"],
    "Alumbrado Público": ["alumbrado", "luz", "poste", "luminaria", "oscuridad", "foco quemado", "farol"],
    "Arbolado Público": ["árbol", "rama", "raíces", "poda", "caído", "tronco", "hojas secas"],
    "Baches y Pavimentación": ["bache", "pozo", "pavimento", "asfalto", "calle rota", "cráter", "hueco", "rompe cubiertas"],
    "Limpieza y Residuos": ["basura", "limpieza", "residuos", "escombros", "mugre", "basural", "suciedad"],
    "Plazas y Parques": ["plaza", "parque", "juegos", "banco", "espacio verde", "columpio", "pasto alto"],
    "Semáforos y Señalización": ["semáforo", "señalización", "cartel", "cruce", "rojo", "indicador", "pare"],
    "Veredas y Accesibilidad": ["vereda", "accesibilidad", "rampa", "baldosa", "peatón", "rotura", "escalón"],
    "Agua y Cloacas": ["agua potable", "cloaca", "caño maestro", "pérdida de agua", "alcantarilla hundida", "desborde cloacal", "olor podrido"],
}
cats = list(semantic_map)
corpus_txt = [f for c_ in cats for f in semantic_map[c_]]
corpus_cat = [c_ for c_ in cats for _ in semantic_map[c_]]

import re
def normalizar_produccion(text):
    t = str(text).lower()
    reemplazos = [
        (r'\bq\b', 'que'), (r'\bx\b', 'por'), (r'\bd\b', 'de'),
        (r'\b(tb|tbn)\b', 'también'), (r'\b(xq|xque)\b', 'porque'),
        (r'\b(vachhe|bacheazo|vache)\b', 'bache'), (r'\b(crter)\b', 'cráter'),
        (r'\balluda\b', 'ayuda'), (r'\b(zemaforo|semaforo)\b', 'semáforo'),
        (r'\b(ranpa)\b', 'rampa'), (r'\b(rrt+o|rrtto|rto)\b', 'roto'),
    ]
    for pat, rep in reemplazos:
        t = re.sub(pat, rep, t)
    return re.sub(r'([a-zA-Z])\1{2,}', r'\1', t)

# --- Agrupación de los reportes válidos según estilo de texto y nivel de ruido ---
# Cada muestra cae en un único grupo, en este orden de prioridad: vago > formal > ruido medio/fuerte > claro/indirecto.
# El grupo 'vago' se reporta aparte porque, por construcción, esas frases no contienen información de categoría
# y sirven como control de validez del experimento (el texto solo no puede, ni debería poder, clasificarlas).
textos_reales = [x for x in dataset if x['expected_category'] != 'Descarte']
casos = {"Claro (sin ruido)": [], "Indirecto (sin anclas)": [], "Formal": [],
         "Ruido medio/fuerte": [], "Vago (no clasificable)": []}
for item in textos_reales:
    desc, cat = item["description"], item["expected_category"]
    estilo, ruido = item.get("text_style", ""), item.get("noise_level", "")
    if estilo == "vago":
        casos["Vago (no clasificable)"].append((desc, cat))
    elif estilo == "formal":
        casos["Formal"].append((desc, cat))
    elif ruido in ("medio", "fuerte"):
        casos["Ruido medio/fuerte"].append((desc, cat))
    elif estilo == "claro":
        casos["Claro (sin ruido)"].append((desc, cat))
    elif estilo == "indirecto":
        casos["Indirecto (sin anclas)"].append((desc, cat))
print({k: len(v) for k, v in casos.items()}, "| total:", sum(len(v) for v in casos.values()))

def argmax_por_categoria(matriz_sims):
    preds = []
    for fila in matriz_sims:
        mejor = {c_: -1.0 for c_ in cats}
        for s_, c_ in zip(fila, corpus_cat):
            mejor[c_] = max(mejor[c_], float(s_))
        preds.append(max(mejor, key=mejor.get))
    return preds

def sims_st(model, textos):
    et = model.encode(textos, convert_to_tensor=True, normalize_embeddings=True)
    ec = model.encode(corpus_txt, convert_to_tensor=True, normalize_embeddings=True)
    return util.cos_sim(et, ec).cpu().numpy()

tfidf_palabra = TfidfVectorizer().fit(corpus_txt)
tfidf_char = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4)).fit(corpus_txt)

def sims_tfidf(vec):
    return lambda model, textos: cosine_similarity(vec.transform(textos), vec.transform(corpus_txt))

configs = [
    ("Sentence-BERT (base BETO) + normalizador", beto_model, sims_st, True),
    ("Sentence-BERT (base BETO) sin normalizador", beto_model, sims_st, False),
    ("MiniLM multilingüe + normalizador", minilm_model, sims_st, True),
    ("MiniLM multilingüe sin normalizador", minilm_model, sims_st, False),
    ("TF-IDF palabra", None, sims_tfidf(tfidf_palabra), False),
    ("TF-IDF n-gramas de caracteres", None, sims_tfidf(tfidf_char), False),
]

filas = []
for nombre, model, fn, norm in configs:
    fila = {"Modelo": nombre}
    for grupo, lista in casos.items():
        if not lista:
            fila[grupo] = "0/0 = 0.0%"
            continue

        textos = [normalizar_produccion(t) if norm else t for t, _ in lista]
        preds = argmax_por_categoria(fn(model, textos))
        k = sum(p == e for p, (_, e) in zip(preds, lista))
        fila[grupo] = fmt_ci(k, len(lista))
    filas.append(fila)

df_nlp = pd.DataFrame(filas)
display(df_nlp)
df_nlp.to_csv(os.path.join(BASE_DIR, "comparativa_nlp_por_estilo.csv"), index=False)


Cargando modelos NLP...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.51k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

{'Claro (sin ruido)': 135, 'Indirecto (sin anclas)': 138, 'Formal': 91, 'Ruido medio/fuerte': 141, 'Vago (no clasificable)': 83} | total: 588


,Modelo,Claro (sin ruido),Indirecto (sin anclas),Formal,Ruido medio/fuerte,Vago (no clasificable)
0,Sentence-BERT (base BETO) + normalizador,92/135 = 68.1% (IC95% 59.9-75.4%),42/138 = 30.4% (IC95% 23.4-38.6%),45/91 = 49.5% (IC95% 39.4-59.5%),56/141 = 39.7% (IC95% 32.0-48.0%),11/83 = 13.3% (IC95% 7.6-22.2%)
1,Sentence-BERT (base BETO) sin normalizador,91/135 = 67.4% (IC95% 59.1-74.7%),42/138 = 30.4% (IC95% 23.4-38.6%),45/91 = 49.5% (IC95% 39.4-59.5%),47/141 = 33.3% (IC95% 26.1-41.5%),12/83 = 14.5% (IC95% 8.5-23.6%)
2,MiniLM multilingüe + normalizador,70/135 = 51.9% (IC95% 43.5-60.1%),32/138 = 23.2% (IC95% 16.9-30.9%),28/91 = 30.8% (IC95% 22.2-40.9%),32/141 = 22.7% (IC95% 16.6-30.3%),7/83 = 8.4% (IC95% 4.1-16.4%)
3,MiniLM multilingüe sin normalizador,69/135 = 51.1% (IC95% 42.8-59.4%),31/138 = 22.5% (IC95% 16.3-30.1%),25/91 = 27.5% (IC95% 19.4-37.4%),29/141 = 20.6% (IC95% 14.7-28.0%),8/83 = 9.6% (IC95% 5.0-17.9%)
4,TF-IDF palabra,105/135 = 77.8% (IC95% 70.1-84.0%),8/138 = 5.8% (IC95% 3.0-11.0%),37/91 = 40.7% (IC95% 31.1-50.9%),34/141 = 24.1% (IC95% 17.8-31.8%),7/83 = 8.4% (IC95% 4.1-16.4%)
5,TF-IDF n-gramas de caracteres,114/135 = 84.4% (IC95% 77.4-89.6%),24/138 = 17.4% (IC95% 12.0-24.6%),39/91 = 42.9% (IC95% 33.2-53.1%),61/141 = 43.3% (IC95% 35.4-51.5%),11/83 = 13.3% (IC95% 7.6-22.2%)


## 3. Búsqueda Espacial: BallTree (Haversine) vs. KDTree vs. fuerza bruta
Se usa como *ground truth* la distancia Haversine exacta, calculada por fuerza bruta. Se comparan tres estrategias de búsqueda de vecinos dentro de un radio: BallTree con métrica Haversine, KDTree sobre coordenadas geográficas crudas (grados, con el radio convertido mediante 111 320 m/°) y KDTree sobre una proyección local en metros (equirrectangular, centrada en la latitud de Mendoza). Se mide tiempo de construcción del índice, tiempo de consulta promedio (200 consultas aleatorias) y falsos negativos/positivos respecto del ground truth. PostGIS no interviene en este experimento; se evalúan únicamente las estructuras de datos de scikit-learn que podrían usarse detrás de una consulta espacial en producción.

In [ ]:
R = 6371000.0
LAT0 = -32.89
N_QUERIES = 200
sizes = [100, 1000, 10000, 50000, 100000]
radios_m = [40, 100]
rng = np.random.default_rng(42)

def haversine_m(lat1, lon1, lat2, lon2):
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi, dl = p2 - p1, np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(np.clip(a, 0.0, 1.0)))

def proyectar(lat, lon):     # metros locales alrededor de LAT0
    return np.column_stack((R * np.radians(lon) * np.cos(np.radians(LAT0)), R * np.radians(lat)))

def cuenta_errores(resultado, verdad):
    fn = sum(len(v - set(r.tolist())) for r, v in zip(resultado, verdad))
    fp = sum(len(set(r.tolist()) - v) for r, v in zip(resultado, verdad))
    return fn, fp

filas = []
for n in sizes:
    lats = rng.uniform(-32.95, -32.85, n); lons = rng.uniform(-68.90, -68.80, n)
    qlat = rng.uniform(-32.95, -32.85, N_QUERIES); qlon = rng.uniform(-68.90, -68.80, N_QUERIES)
    coords_rad = np.radians(np.column_stack((lats, lons))); q_rad = np.radians(np.column_stack((qlat, qlon)))
    coords_deg = np.column_stack((lats, lons)); q_deg = np.column_stack((qlat, qlon))
    X, Q = proyectar(lats, lons), proyectar(qlat, qlon)

    t0 = time.perf_counter(); ball = BallTree(coords_rad, metric="haversine"); b_ball = (time.perf_counter() - t0) * 1000
    t0 = time.perf_counter(); kd_deg = KDTree(coords_deg); b_kd_deg = (time.perf_counter() - t0) * 1000
    t0 = time.perf_counter(); kd_m = KDTree(X); b_kd_m = (time.perf_counter() - t0) * 1000

    for r in radios_m:
        t0 = time.perf_counter()
        verdad = [set(np.where(haversine_m(qlat[i], qlon[i], lats, lons) <= r)[0].tolist()) for i in range(N_QUERIES)]
        t_brute = (time.perf_counter() - t0) / N_QUERIES * 1000
        total_vecinos = sum(len(v) for v in verdad)

        t0 = time.perf_counter(); res_ball = ball.query_radius(q_rad, r=r / R); t_ball = (time.perf_counter() - t0) / N_QUERIES * 1000
        t0 = time.perf_counter(); res_kd_deg = kd_deg.query_radius(q_deg, r=r / 111320.0); t_kd_deg = (time.perf_counter() - t0) / N_QUERIES * 1000
        t0 = time.perf_counter(); res_kd_m = kd_m.query_radius(Q, r=r); t_kd_m = (time.perf_counter() - t0) / N_QUERIES * 1000

        fn_b, fp_b = cuenta_errores(res_ball, verdad)
        fn_d, fp_d = cuenta_errores(res_kd_deg, verdad)
        fn_m, fp_m = cuenta_errores(res_kd_m, verdad)
        filas.append({
            "N": n, "radio (m)": r, "vecinos reales": total_vecinos,
            "build Ball (ms)": b_ball, "build KD grados (ms)": b_kd_deg, "build KD metros (ms)": b_kd_m,
            "query Ball (ms)": t_ball, "query KD grados (ms)": t_kd_deg, "query KD metros (ms)": t_kd_m, "query brute (ms)": t_brute,
            "FN Ball": fn_b, "FN KD grados": fn_d, "FN KD metros": fn_m,
            "FP Ball": fp_b, "FP KD grados": fp_d, "FP KD metros": fp_m,
        })
df_espacial = pd.DataFrame(filas)
display(df_espacial.round(3))
df_espacial.to_csv(os.path.join(BASE_DIR, "comparativa_espacial.csv"), index=False)
print(f"A 33°S, 1° de longitud mide {111320*np.cos(np.radians(33))/1000:.1f} km vs {111.32:.1f} km de latitud "
      f"(contracción {100*(1-np.cos(np.radians(33))):.1f}%). Un KDTree en grados con radio isótropo subcubre la dirección E-O; "
      "un KDTree en metros locales o un BallTree Haversine no.")
print("Nota: el ground truth es Haversine (Tierra esférica); frente a un elipsoide la diferencia es <0.5% a estas escalas.")


,N,radio (m),vecinos reales,build Ball (ms),build KD grados (ms),build KD metros (ms),query Ball (ms),query KD grados (ms),query KD metros (ms),query brute (ms),FN Ball,FN KD grados,FN KD metros,FP Ball,FP KD grados,FP KD metros
0,100,40,2,6.417,5.006,0.198,0.017,0.002,0.002,0.057,0,2,0,0,0,0
1,100,100,6,6.417,5.006,0.198,0.007,0.002,0.002,0.055,0,0,0,0,0,0
2,1000,40,10,0.713,0.455,0.514,0.010,0.004,0.004,0.126,0,1,0,0,0,0
3,1000,100,73,0.713,0.455,0.514,0.009,0.004,0.004,0.129,0,11,0,0,0,0
4,10000,40,79,8.270,4.451,4.429,0.013,0.006,0.006,0.810,0,13,0,0,0,0
5,10000,100,597,8.270,4.451,4.429,0.015,0.006,0.006,0.781,0,92,0,0,0,0
6,50000,40,454,58.123,29.756,29.273,0.019,0.010,0.010,3.213,0,62,0,0,0,0
7,50000,100,3041,58.123,29.756,29.273,0.016,0.008,0.008,2.788,0,481,0,0,0,0
8,100000,40,997,75.167,45.297,55.249,0.017,0.008,0.007,5.354,0,152,0,0,0,0
9,100000,100,6005,75.167,45.297,55.249,0.022,0.010,0.010,3.808,0,971,2,0,0,1


A 33°S, 1° de longitud mide 93.4 km vs 111.3 km de latitud (contracción 16.1%). Un KDTree en grados con radio isótropo subcubre la dirección E-O; un KDTree en metros locales o un BallTree Haversine no.
Nota: el ground truth es Haversine (Tierra esférica); frente a un elipsoide la diferencia es <0.5% a estas escalas.
